## 2.5 向量存储(Vector Stores)
将文本向量化之后，下一步就是进行向量的存储。
### 2.5.1 向量数据库的理解
假设你是一名摄影师，拍了大量的照片。为了方便管理和查找，你决定将这些 照片存储 到一个数据库中。传统的 关系型数据库 （如 MySQL、PostgreSQL 等）可以帮助你 存储照片的元数据 ，比如拍摄时间、地点、参数信息等。

但是，当你想要根据 照片的内容（如颜色、纹理、物体等） 进行搜索时，传统数据库将无法满足你的需求，因为它们通常以数据表的形式存储数据，并使用查询语句进行精确搜索。那么此时，向量数据库就可以派上用场。

我们可以构建一个多维的空间使得每张照片特征都存在于这个空间内，并用已有的维度进行表示，比如时间、地点、相机型号、颜色....此照片的信息将作为一个点，存储于其中。以此类推，即可在该空间中构建出无数的点，而后我们将这些点与空间坐标轴的原点相连接，就成为了一条条向量，当这些点变为向量之后，即可利用向量的计算进一步获取更多的信息。当要进行照片的检索时，也会变得更容易更快捷。

注意：在向量数据库中进行检索时，检索并 不是唯一的、精确的 ，而是查询和目标向量 最为相似的一些向量 ，具有模糊性延伸思考：只要对图片、视频、商品等素材进行向量化，就可以实现以图搜图、视频相关推荐、相似宝贝推荐等功能。

这里的“不精确”有两层意思：

1. **语义上**：返回的是和问题“最像”的 k 条，不保证其中有一条正好是答案；
2. **算法上**：数据量大时，向量库一般不会把查询向量和库里每一条都算一遍相似度（这叫 FLAT，暴力检索，结果精确但慢），而是预先建一个**近似最近邻（ANN）索引**，比如 HNSW、IVF，检索时只比较一部分候选，快很多，代价是偶尔漏掉真正最近的几条。

2.5.3 的 `create_collection` 会自动给向量字段建这样的索引。

### 2.5.2 常用的向量数据库
LangChain提供了众多向量存储的集成，包括开源的本地向量存储与云托管的私有向量存储。并公开了一个标准接口，可以轻松地在向量存储之间进行交换。

常用向量数据库:

| 数据库 | 类型 | 特点 | LangChain 集成包 / 类 |
|---|---|---|---|
| ⭐ Milvus | 开源，专门的向量数据库 | 有 Lite（本地文件）、Standalone、分布式三种部署，代码一样；索引类型多，支持标量过滤、混合检索。托管版是 Zilliz Cloud | `langchain-milvus` / `Milvus` |
| Chroma | 开源，可以嵌入在 Python 进程里运行 | 安装简单，适合原型 | `langchain-chroma` / `Chroma` |
| Qdrant | 开源，Rust 编写 | 元数据过滤能力强，也有本地模式 | `langchain-qdrant` / `QdrantVectorStore` |
| Weaviate | 开源 | 内置混合检索 | `langchain-weaviate` / `WeaviateVectorStore` |
| ⭐ PGVector | PostgreSQL 的扩展 | 向量和业务数据放在同一个库里，可以直接写 SQL | `langchain-postgres` / `PGVector`、`PGVectorStore` |
| Elasticsearch / OpenSearch | 搜索引擎加上向量检索 | 关键词检索和向量检索都能做 | `langchain-elasticsearch` 等 |
| Redis | 内存数据库 | 延迟低 | `langchain-redis` |
| Pinecone | 全托管云服务 | 不用自己运维 | `langchain-pinecone` |
| FAISS | Meta 开源的向量检索**库**，不是数据库 | 只负责建索引和检索，没有服务端，持久化要自己存文件 | `langchain-community` / `FAISS` |
| ⭐ InMemoryVectorStore | LangChain 自带，数据放在内存里 | 不用安装任何东西，只适合学习和测试 | `langchain-core` / `InMemoryVectorStore` |

⭐：本节用的 Milvus，2.5.4 对比用的 InMemoryVectorStore，以及业务数据已经在 PostgreSQL 里时常用的 PGVector。

> ⚠️ **与旧教程不同**：旧教程里 Chroma、FAISS、Milvus 等都从 `langchain_community.vectorstores` 导入。现在各数据库都有自己的集成包（上表最后一列）。`langchain-community` 已在 2026-05-22 停止维护，FAISS 的集成只留在这个包里，官方文档的 FAISS 页面也已经打不开（404）。

### 2.5.3 Milvus数据库
本小节我们使用 Milvus-lite 数据库演示。

#### 2.5.3.1 整体结构
从业务角度，Milvus数据模型层级如下
```python
Database -> Collection -> Partition -> Entity
```

* Database：Milvus 的数据库，用来隔离不同业务数据。
* Collection：最核心的逻辑容器，类似于关系型数据库里的 table。
* Partition：分区，是 collection 的子集，不是必须手动创建；一个 collection 至少会有默认partition。
* Entity：可以理解为 collection 中的一条记录。类似于关系型数据库的一行数据。

除了这四层，每个 collection 还有自己的 **schema**（有哪些字段）、**索引**和**加载状态**，见 2.5.3.3 的 `create_collection`。

#### 2.5.3.2 具体说明
在传统数据库中，如果你想存储用户信息，你会建一张表叫 user_table ；想存储商品，会建一张表叫product_table 。

在向量数据库中，我们要存储的是成千上万个“特征向量”以及它们对应的文本。为了把不同用途的向量分门别类地存放，我们就要创建不同的 Collection。 比如， COLLECTION_NAME = "docs" 就是给这张“表”取名叫 docs ，专门用来存放你切分好的文档片段向量。

#### 2.5.3.3 数据库DDL演示

In [1]:
MILVUS_URI = "db/rag.db" # Milvus 服务的连接地址
DB_NAME = "rag_tutorial" # 自定义数据库名称
COLLECTION_NAME = "docs" # 向量集合名称（类似于传统数据库的表）
KNOWLEDGE_FILE = "knowledge.txt" # 本地知识库文件路径

In [ ]:
from pymilvus import MilvusClient
client = MilvusClient(f"db/{DB_NAME}.db")
existed_databases = client.list_databases()

**`MilvusClient(...)` 的参数**（pymilvus 3.0.2，`pymilvus/milvus_client/milvus_client.py`）：

| 参数 | 默认值 | 含义 |
|---|---|---|
| `uri` | `"http://localhost:19530"` | 连接地址，决定连哪种 Milvus（见下表） |
| `user`、`password` | `""` | 服务端开了认证时的用户名、密码，等价于 `token="user:password"` |
| `token` | `""` | 认证令牌，Zilliz Cloud 用 API Key |
| `db_name` | `""` | 连上后使用的数据库，空字符串表示 `default` |
| `timeout` | `None` | 每次调用的超时时间（秒） |

Milvus 的几种部署方式，代码都一样，只是 `uri` 不同：

| 部署方式 | `uri` 写法 | 适合 |
|---|---|---|
| Milvus Lite | 本地路径，以 `.db` 结尾，如 `"db/rag_tutorial.db"` | 学习、原型，数据量小 |
| Standalone | `"http://localhost:19530"`（用 Docker 在单机运行） | 测试、中小规模 |
| Distributed | 集群的地址 | 生产、大规模 |
| Zilliz Cloud | 云端地址，加上 `token` | 托管服务 |

`uri` 以 `.db` 结尾时，pymilvus（`ConnectionConfig.from_uri`）先在本机启动一个 Milvus Lite 服务，再通过 gRPC 连上它。所以：

* 上面定义的 `MILVUS_URI = "db/rag.db"` 没有用到，实际连接的是 `db/rag_tutorial.db`。在 milvus-lite 3.x 里它是一个**文件夹**，里面是预写日志（WAL）和数据文件；
* 输出里的 `GOAWAY ... too_many_pings` 来自这个本地 gRPC 连接：客户端每 10 秒发一次心跳，Lite 服务端嫌太频繁，于是回了这条消息，客户端把间隔加倍（输出里 10000ms、20000ms、40000ms…）。它不影响结果。

> ⚠️ **与官方文档不同**：官方的 Milvus Lite 页面写着“只支持 FLAT 索引”“不支持 partition”。本项目装的 milvus-lite 3.2.1 是用 Python 重写的版本，源码（`milvus_lite/index/factory.py`）里 `AUTOINDEX` 在装了 faiss-cpu 时用 HNSW 索引，partition 也能创建和检索（实测）。不过它仍然只适合小数据量，一致性级别也只支持 Strong。

创建新的数据库

In [5]:
if DB_NAME not in existed_databases:
    client.create_database(db_name=DB_NAME)
client.list_databases()

['default', 'rag_tutorial']

I1010 01:15:50.091510 72433022 chttp2_transport.cc:1400] ipv4:127.0.0.1:60002: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1010 01:15:50.091523 72433022 chttp2_transport.cc:1432] ipv4:127.0.0.1:60002: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


**数据库的四个方法**：

| 方法 | 参数 | 返回值 | 说明 |
|---|---|---|---|
| `list_databases(timeout=None)` | 无 | `list[str]` | 所有数据库的名字，总有一个 `default` |
| `create_database(db_name, properties=None, timeout=None)` | 库名；`properties` 是库级别的配置 | `None` | 库已经存在会报错 `database '...' already exists`，所以上面先判断了一下 |
| `use_database(db_name)` | 库名 | `None` | 切换**这个客户端**之后操作的库。也可以在创建客户端时传 `db_name` |
| `drop_database(db_name)` | 库名 | `None` | 库里还有 collection 会报错 `database '...' is not empty`；`default` 库不能删 |

这些方法成功时都返回 `None`，失败时抛出 `MilvusException`。

删除数据库

如果数据库下有Collection则无法删除，需要先删除它的所有Collection才能删除Database

In [ ]:
client.drop_database('default')

DropDatabase failed: cannot drop default database
Traceback (most recent call last):
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/milvus_lite/adapter/grpc/servicer.py", line 1104, in DropDatabase
    self._db.drop_database(request.db_name)
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/milvus_lite/db.py", line 147, in drop_database
    raise ValueError("cannot drop default database")
ValueError: cannot drop default database
2026-10-10 01:20:47,614 [ERROR][_log_rpc_error]: RPC error: [drop_database], <MilvusException: (code=1, message=cannot drop default database)>, <elapsed:2.2ms>
Traceback:
Traceback (most recent call last):
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/pymilvus/decorators.py", line 518, in handler
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/pymilvus/decorators.py

MilvusException: <MilvusException: (code=1, message=cannot drop default database)>

上面的报错是 Lite 服务端拒绝删除 `default` 库（`cannot drop default database`）。pymilvus 先把错误打印成一大段日志，再抛出 `MilvusException`，所以输出里同一个错误出现了两次。

Collection操作

① 切换数据库

In [7]:
client.use_database(db_name=DB_NAME)

② 查看数据库下的collections

In [10]:
collections = client.list_collections()

for coll in collections:
    print(coll)

docs


I1010 01:25:38.990101 72433014 chttp2_transport.cc:1400] ipv4:127.0.0.1:60002: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1010 01:25:38.990121 72433014 chttp2_transport.cc:1432] ipv4:127.0.0.1:60002: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms


③ 创建collection

In [9]:
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=2048,
    metric_type="COSINE"
)

源码剖析
```python
    def create_collection(
        self,
        collection_name: str,
        dimension: Optional[int] = None,
        primary_field_name: str = "id",  # default is "id"
        id_type: str = "int",  # or "string",
        vector_field_name: str = "vector",  # default is  "vector"
        metric_type: str = "COSINE",
        auto_id: bool = False,
        timeout: Optional[float] = None,
        schema: Optional[CollectionSchema] = None,
        index_params: Optional[IndexParams] = None,
        **kwargs,
    ):
```

签名和当前源码（pymilvus 3.0.2）一致。参数：

| 参数 | 默认值 | 含义 |
|---|---|---|
| `collection_name` | 必填 | collection 的名字 |
| `dimension` | `None` | 向量维度。快速模式（不传 `schema`）必填，**必须等于嵌入模型输出的维度**：这里用 jina-embeddings-v4，是 2048。维度不对，写入时会报 `expected dim 2048, got ...` |
| `primary_field_name` | `"id"` | 主键字段的名字 |
| `id_type` | `"int"` | 主键类型：`"int"` 是 INT64；`"string"` 是 VARCHAR，这时还要传 `max_length` |
| `vector_field_name` | `"vector"` | 向量字段的名字 |
| `metric_type` | `"COSINE"` | 相似度怎么算：`"COSINE"`（余弦）、`"L2"`（欧氏距离）、`"IP"`（内积），见 2.5.3.5 |
| `auto_id` | `False` | `True` 时由 Milvus 生成主键，写入的数据里不用带 `id` |
| `timeout` | `None` | 超时时间（秒） |
| `schema` | `None` | 传了就走**自定义模式**，上面的 `dimension`、`metric_type` 等快速参数都不再起作用 |
| `index_params` | `None` | 自定义模式下的索引参数，用 `client.prepare_index_params()` 生成 |
| `**kwargs` | | 例如 `enable_dynamic_field`（动态字段，快速模式默认 `True`）、`consistency_level`（一致性级别） |

**返回值**：`None`。

只传 `dimension` 时，源码 `_fast_create_collection` 一次做了四件事：

![create_collection 快速模式](assets/13-milvus-quick-setup.svg)

其中“一致性级别”决定刚写入的数据多久能被查到：`Strong` 立刻能查到，服务端的默认值 `Bounded` 可能要等一小会儿。

第 ④ 步也要留意：collection 只有在“已加载”状态下才能 `search`、`get`、`query`。重新连接已有的数据库时（比如重启 kernel 后跳过 `create_collection` 直接查询），collection 处于 released 状态，查询会报 `Collection 'docs' is in state 'released'; call load() before search/get/query`（实测），要先调用 `client.load_collection(COLLECTION_NAME)`。当前状态可以用 `client.get_load_state(COLLECTION_NAME)` 查看。

④ 删除collection


In [11]:
client.drop_collection(collection_name=COLLECTION_NAME)

In [12]:
collections = client.list_collections()

for coll in collections:
    print(coll)

I1010 01:34:42.881322 72433015 chttp2_transport.cc:1400] ipv4:127.0.0.1:60002: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1010 01:34:42.881339 72433015 chttp2_transport.cc:1432] ipv4:127.0.0.1:60002: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 40000ms


* `list_collections()` 返回当前数据库里所有 collection 的名字（`list[str]`），删掉以后列表为空，所以上面什么也没打印；
* `drop_collection(collection_name, timeout=None)` 返回 `None`，会连同数据和索引一起删掉，不能恢复。名字不存在时不报错；
* 想先判断再操作，用 `has_collection(collection_name)`，返回 `True` / `False`。

#### 2.5.3.4 DML 操作
准备嵌入模型

In [13]:
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")
embedding_model = OpenAIEmbeddings(
    model="jina-embeddings-v4",
    api_key=os.environ["embedding_model_key"],
    base_url=os.environ["embedding_model_base"],
    check_embedding_ctx_length=False,  # 非 OpenAI 模型：直接发原文，不发 token id，因为不同模型的词表可能不同
)

准备collection

In [14]:
# 1. 创建
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=2048,
    metric_type="COSINE"
)

In [15]:
# 2. 查询元数据
from rich import print as rprint

metadata = client.describe_collection(collection_name=COLLECTION_NAME)

rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 0,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 0,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 2048}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 0,
    'consistency_level': 0,
    'consistency_level_name': 'Strong',
    'properties': {},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'schema_version': 0
}

`describe_collection(collection_name)` 返回一个字典，描述这个 collection。几个重要的键：

| 键 | 这里的值 | 含义 |
|---|---|---|
| `fields` | `id`（INT64，主键）、`vector`（FLOAT_VECTOR，`dim` 2048） | 快速模式建的两个字段 |
| `auto_id` | `False` | 主键由我们自己给 |
| `enable_dynamic_field` | `True` | 打开了动态字段：schema 里没有的键（后面的 `text`、`source`）存进隐藏的 `$meta` 字段，所以 `fields` 里看不到它 |
| `consistency_level_name` | `'Strong'` | 一致性级别。Milvus Lite 只支持 Strong |
| `num_partitions` | `1` | 只有一个默认分区 `_default` |

准备数据

In [16]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

# 生成嵌入向量
vectors = embedding_model.embed_documents(texts=texts)

In [17]:
print(len(vectors))
print(len(vectors[0]))
print(vectors[0][:10])

4
2048
[0.04327745735645294, -0.019777439534664154, -0.006889113690704107, -0.029114624485373497, -0.0036037294194102287, -0.003207771573215723, 0.0454728826880455, -0.004979606252163649, 0.01703583635389805, -0.004272987134754658]


I1010 01:45:41.401904 72433020 chttp2_transport.cc:1400] ipv4:127.0.0.1:60002: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1010 01:45:41.401919 72433020 chttp2_transport.cc:1432] ipv4:127.0.0.1:60002: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 80000ms


④ 封装为可以插入的数据格式

由上可知，collection需要 id字段 作为主键唯一标识数据，需要 vector字段 存储嵌入向量，其余字段
可以按需添加。

In [18]:
data = [
    {
        "id" : i,
        "vector" : vectors[i],
        "text" : texts[i],
        "source" : "demo"
    } for i in range(len(texts))
]

写入的数据是**字典的列表**，一个字典是一条 entity，键要和字段对上（见上面的图）：

* `id`：主键，这里用整数，因为快速模式的 `id_type` 默认是 `"int"`；
* `vector`：列表长度必须等于 `dimension`（2048）；
* `text`、`source`：schema 里没有，进入动态字段 `$meta`。原文要自己存一份（这里是 `text`），检索时才能拿回文本，向量本身还原不出原文。

关掉动态字段（`enable_dynamic_field=False`）时，多出来的键会报 `DataNotMatchException: Attempt to insert an unexpected field`。

写入数据

1. 插入数据

In [ ]:
insert_res = client.upsert(
    collection_name=COLLECTION_NAME,
    data=data,
) 

print("insert result : ",insert_res)

insert result :  {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


upsert可以保证幂等写入，即主键相同时覆盖

**`insert` 和 `upsert` 的参数一样**：

| 参数 | 默认值 | 含义 |
|---|---|---|
| `collection_name` | 必填 | 写到哪个 collection |
| `data` | 必填 | 一个字典或字典的列表 |
| `partition_name` | `""` | 写到哪个分区，空表示 `_default` |
| `timeout` | `None` | 超时时间（秒） |
| `**kwargs` | | `upsert` 还可以传 `partial_update=True`，只更新给出的字段 |

**返回值**：`insert` 返回 `{'insert_count': 条数, 'ids': [主键…]}`，`upsert` 返回 `{'upsert_count': 条数, 'ids': [主键…]}`（上面的输出）。

**两者的区别在主键已经存在时**：

* `upsert`：官方文档说它是“插入新的 + 删除旧的”，所以重复执行结果一样（幂等）；
* `insert`：官方文档说它**不检查主键是否重复**，同一个主键会出现两条数据。milvus-lite 3.2.1 实测是覆盖旧数据，和服务端不一样，不要依赖这个行为。

重复入库（比如重跑 notebook）时用 `upsert`。

② 手动flush

Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘

> ⚠️ **上面的说法不准确**：
> * **能不能查到刚写入的数据，看一致性级别，不看 flush**。`Strong` 下写完就能 `search`、`query` 到；
> * pymilvus 源码对 `flush` 的说明是“把所有 segment 封存（seal），之后写入的数据进新的 segment，只有封存的 segment 才能建索引”。官方 API 文档还提醒，频繁手动 flush 会产生很多小 segment，影响检索性能，一般交给 Milvus 自动处理；
> * 服务端的 `get_collection_stats` 统计的行数**不是实时的**（pymilvus 里 `num_entities` 的注释：insert 之后是 0，flush 之后才是 2），这才是“flush 之后才能看到”的那部分；
> * milvus-lite 3.2.1 的写入先记到磁盘上的 WAL 日志，再放进内存表；内存表满 10000 条时自动 flush 成数据文件。所以在 Lite 里不 flush 数据也不会丢，统计的行数也是实时的（实测）。

In [20]:
client.flush(collection_name=COLLECTION_NAME)

③ 查看collection统计信息

In [21]:
stats = client.get_collection_stats(collection_name=COLLECTION_NAME)

print("stats : ",stats)

stats :  {'row_count': 4}


I1010 02:09:30.173029 72433014 chttp2_transport.cc:1400] ipv4:127.0.0.1:60002: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1010 02:09:30.173049 72433014 chttp2_transport.cc:1432] ipv4:127.0.0.1:60002: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 160000ms


`get_collection_stats(collection_name, timeout=None)` 返回字典 `{'row_count': 行数}`。服务端的这个数字不是实时的（上面的 ⚠️）；要准确的行数，用 `query` 加 `count(*)`（下面 DQL 的“按条件查询”）。

#### 2.5.3.5 DQL操作
1. 扫描数据

通过 query_iterator 扫描collection下的所有数据

In [23]:
iterator = client.query_iterator(
    collection_name=COLLECTION_NAME,
    batch_size=100,
    filter="",
    output_fields=["*"]
)

i = 0
while True:

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"第{i + 1}条数据：")
        # print(row)

        print(f"id : {row['id']}, vector = {row['vector'][:5]}, text = {row['text']}, source = {row['source']}")

        i += 1

iterator.close()

2026-10-10 02:13:26,958 [WARNING][__setup_ts_by_request]: failed to get mvccTs from milvus server, use client-side ts instead (query_iterator.py:306)


第1条数据：
id : 0, vector = [0.04327745735645294, -0.019777439534664154, -0.006889113690704107, -0.029114624485373497, -0.0036037294194102287], text = LangChain 是一个用于构建 LLM 应用的开发框架。, source = demo
第2条数据：
id : 1, vector = [0.0209363903850317, 0.022567423060536385, 0.024888036772608757, -0.010465993545949459, -0.03316522017121315], text = Milvus 是一个适合 AI 应用的向量数据库。, source = demo
第3条数据：
id : 2, vector = [0.03422592952847481, 0.0058259302750229836, -0.016337456181645393, -0.0007080587092787027, 0.0011546994792297482], text = RAG 的核心是先检索相关知识，再让大模型生成答案。, source = demo
第4条数据：
id : 3, vector = [-0.002095661824569106, 0.013315911404788494, 0.020677901804447174, -0.035465992987155914, -0.03536370024085045], text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。, source = demo


**`query_iterator` 的参数**：

| 参数 | 默认值 | 含义 |
|---|---|---|
| `collection_name` | 必填 | |
| `batch_size` | `1000` | 每次 `next()` 最多取多少条 |
| `limit` | 不限 | 一共最多取多少条 |
| `filter` | `""` | 过滤表达式，空字符串表示不过滤（语法见下面的“按条件查询”） |
| `output_fields` | `None` | 要返回的字段。`["*"]` 表示全部字段，包括向量和动态字段里的键 |
| `partition_names` | `None` | 只在这些分区里查 |
| `timeout` | `None` | 超时时间（秒） |

**返回值**：一个 `QueryIterator` 对象。每调用一次 `next()` 返回一批 `list[dict]`，取完后返回空列表，所以上面用 `if not rows: break` 结束；用完调用 `close()`。

为什么要用迭代器：一次 `query` 的 `offset + limit` 要小于 16384（Zilliz 的 API 文档），数据多时只能分批取。

输出开头的 `failed to get mvccTs from milvus server, use client-side ts instead` 是警告：迭代器想向服务端要一个时间戳，保证每一批看到的数据一致，Lite 不提供，就改用客户端的时间。不影响结果。

2. 通过主键查询数据

In [25]:
res = client.get(collection_name=COLLECTION_NAME,ids=[1,2,3])

print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id : {res[i]['id']},vector = {res[i]['vector'][:5]},text = {res[i]['text']},source = {res[i]['source']}")
    # print(res[i])


3
第1条数据：
id : 1,vector = [0.0209363903850317, 0.022567423060536385, 0.024888036772608757, -0.010465993545949459, -0.03316522017121315],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第2条数据：
id : 2,vector = [0.03422592952847481, 0.0058259302750229836, -0.016337456181645393, -0.0007080587092787027, 0.0011546994792297482],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo
第3条数据：
id : 3,vector = [-0.002095661824569106, 0.013315911404788494, 0.020677901804447174, -0.035465992987155914, -0.03536370024085045],text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source = demo


**`get(collection_name, ids, output_fields=None, timeout=None, partition_names=None)`**：按主键取数据。

* `ids` 可以是一个主键，也可以是列表；
* 返回 `list[dict]`，每个字典是一条数据。不存在的主键直接跳过，不报错（实测 `ids=[1, 99]` 只返回 1 条）；
* 不传 `output_fields` 时，上面的输出包含了全部字段。

**补充：按条件查询（`query`）**

`get` 只能按主键查。按其他字段的条件查，用 `query(collection_name, filter="", output_fields=None, timeout=None, ids=None, partition_names=None, **kwargs)`，返回 `list[dict]`。`kwargs` 里常用的是 `limit`、`offset`。

In [ ]:
for expr in ['id in [1, 2]', 'text like "Milvus%"', 'source == "demo" and id > 1']:
    res = client.query(collection_name=COLLECTION_NAME, filter=expr, output_fields=["text"])
    print(f"{expr:30} → {[row['id'] for row in res]}")

# 统计行数：output_fields 里写 count(*)
print(client.query(collection_name=COLLECTION_NAME, filter="", output_fields=["count(*)"]))

id in [1, 2]                   → [1, 2]
text like "Milvus%"            → [1]
source == "demo" and id > 1    → [2, 3]
data: ["{'count(*)': 4}"], extra_info: {}


`filter` 是一个字符串表达式，`search`、`query`、`query_iterator`、`delete` 都用同一套语法：

| 写法 | 例子 |
|---|---|
| 比较 | `id == 1`、`id != 1`、`id > 1`、`id <= 2` |
| 在 / 不在列表里 | `id in [1, 2]`、`id not in [1, 2]` |
| 字符串前缀匹配 | `text like "Milvus%"` |
| 组合 | `source == "demo" and id > 1`、`not (id == 0)`、`… or …` |
| 动态字段 | 直接写键名 `source == "demo"`，也可以写成 `$meta["source"] == "demo"` |

字符串要加双引号，所以整个表达式外面用单引号。

3. 相似度查询

① 准备查询嵌入

In [26]:
# 相似度检索
query = "什么是langchain？"
query_vector = embedding_model.embed_query(query)
print(query_vector[:10])

[0.043474748730659485, -0.011614928022027016, -0.010459933429956436, -0.032195672392845154, -0.0037484068889170885, -0.012971022166311741, 0.04108830541372299, 0.006784157827496529, -0.02142009325325489, -0.002138514304533601]


② 检索

In [32]:
result = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    limit=3,
    output_fields=["text","id","distance","source"]
)
print(type(result))
print(len(result))

<class 'pymilvus.client.search_result.SearchResult'>
1


**`search` 的参数**（只列常用的）：

| 参数 | 默认值 | 含义 |
|---|---|---|
| `collection_name` | 必填 | |
| `data` | `None` | **查询向量的列表**，所以单个向量也要写成 `[query_vector]`。可以一次传多个向量，每个向量各返回一组结果 |
| `limit` | `10` | 每个查询向量返回几条，也就是 top-k |
| `filter` | `""` | 先按条件过滤，再在剩下的数据里找最相似的，语法同上 |
| `output_fields` | `None` | 结果里带哪些字段 |
| `search_params` | `None` | 索引的检索参数，例如 HNSW 的 `{"params": {"ef": 64}}`、IVF 的 `{"params": {"nprobe": 10}}`；`AUTOINDEX` 一般不用传 |
| `anns_field` | `None` | 在哪个向量字段上检索。只有一个向量字段时不用传 |
| `partition_names` | `None` | 只在这些分区里检索 |
| `timeout` | `None` | 超时时间（秒） |

**返回值**：`SearchResult`，可以当成 `list[list[dict]]` 用。外层每个元素对应 `data` 里的一个查询向量，所以 `len(result)` 是 1。

In [ ]:
for i in result[0]:
    rprint(i)

{
    'id': 0,
    'distance': 0.8585225343704224,
    'entity': {'id': 0, 'text': 'LangChain 是一个用于构建 LLM 应用的开发框架。', 'source': 'demo'}
}

{
    'id': 2,
    'distance': 0.5008307099342346,
    'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}
}

{
    'id': 1,
    'distance': 0.4678635597229004,
    'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}
}

I1010 02:35:37.213712 72433025 chttp2_transport.cc:1400] ipv4:127.0.0.1:60002: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1010 02:35:37.213735 72433025 chttp2_transport.cc:1432] ipv4:127.0.0.1:60002: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 320000ms


`result[0]` 是第一个查询向量的结果，里面每一条是一个字典：

| 键 | 含义 |
|---|---|
| `id` | 主键，总是会返回 |
| `distance` | 和查询向量的“距离”，含义由 `metric_type` 决定（见下表） |
| `entity` | `output_fields` 里要的字段 |

所以 `output_fields` 里的 `"id"`、`"distance"` 是多余的：`id` 本来就会返回，`distance` 也不是字段，被忽略了（`entity` 里没有它）。

`distance` 这个名字容易误解，它的含义取决于度量：

| `metric_type` | `distance` 是什么 | 排序 |
|---|---|---|
| `COSINE` | 余弦**相似度**，-1 到 1 | 越大越像，从大到小排 |
| `IP` | 内积 | 越大越像 |
| `L2` | 欧氏距离的**平方**（Milvus 不开根号） | 越小越像，从小到大排 |

这里是 `COSINE`：“什么是langchain？”和第 0 条的相似度 0.86，排第一；和另外两条只有 0.50、0.47。

另外，pymilvus 的源码注释说不传 `output_fields` 时只返回主键和距离；milvus-lite 3.2.1 实测会返回全部字段（包括 2048 维的向量），所以最好明确写出要哪些字段。

下面一次传两个查询向量，同时加上过滤条件：

In [ ]:
result = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector, vectors[3]],         # 两个查询向量：问题、第 3 条文本自己的向量
    limit=2,
    filter='source == "demo" and id != 0',   # 先排除 id 为 0 的数据，再找最相似的
    output_fields=["text"],
)

print(len(result))
for i, hits in enumerate(result):
    print(f"第 {i + 1} 个查询向量：")
    for hit in hits:
        print(f"  id={hit['id']}  distance={hit['distance']:.4f}  {hit['entity']['text']}")

2
第 1 个查询向量：
  id=2  distance=0.5008  RAG 的核心是先检索相关知识，再让大模型生成答案。
  id=1  distance=0.4679  Milvus 是一个适合 AI 应用的向量数据库。
第 2 个查询向量：
  id=3  distance=1.0000  Docker Desktop 可以方便地在本地运行 Milvus Standalone。
  id=1  distance=0.7170  Milvus 是一个适合 AI 应用的向量数据库。


* 外层有 2 组结果，和 `data` 里的两个向量一一对应；
* 第 1 组：id 0 被过滤掉了，只能从剩下的里面挑；
* 第 2 组：用第 3 条文本自己的向量去查，排第一的就是它自己，相似度约为 1。

**补充：删除数据（`delete`）**

`delete(collection_name, ids=None, timeout=None, filter=None, partition_name=None)`：按主键或按条件删除，`ids` 和 `filter` 只能二选一。放在 DQL 后面演示，这样前面的查询都还能看到 4 条数据：

In [ ]:
def count_rows():
    # 当 output_fields=["count(*)"] 且 filter=""（无条件）时，query 不会真的返回数据行，而是只返回一行、且这行只有一个字段：
    # {"count(*)": 3} 
    return client.query(collection_name=COLLECTION_NAME, filter="", output_fields=["count(*)"])[0]["count(*)"]

print("按主键删除：", client.delete(collection_name=COLLECTION_NAME, ids=[3]), "剩余", count_rows())
print("按条件删除：", client.delete(collection_name=COLLECTION_NAME, filter="id >= 2"), "剩余", count_rows())

# 把删掉的两条写回去，恢复成 4 条
client.upsert(collection_name=COLLECTION_NAME, data=[data[2], data[3]])
print("写回后：", count_rows())

按主键删除： [3] 剩余 3
按条件删除： [2] 剩余 2
写回后： 4


**返回值**：milvus-lite 3.2.1 返回被删除的主键列表（上面的 `[3]`、`[2]`）。pymilvus 源码说明，Milvus 服务端从 2.3.2 起不再返回主键，这时返回的是 `{'delete_count': 条数}`。所以判断删除成功，看有没有抛出异常，不要依赖返回值的格式。

### 2.5.4 其他向量数据库的通用方法和参数

不同向量数据库的客户端 API 各不相同，但做的是同一套事情：

建集合（定维度、相似度度量、索引）→ 写入（向量 + 原文 + 元数据）→ 按 id 取 / 按条件查 → 相似度检索（top-k + 过滤）→ 删除

#### 2.5.4.1 通用参数，以及几个数据库的原生方法

**通用参数**：

| 参数 | 作用 | Milvus 里的名字 |
|---|---|---|
| 维度 | 向量的长度，建集合时定下来，必须等于嵌入模型输出的维度 | `dimension` |
| 相似度度量 | 余弦、欧氏距离、内积三选一，要和嵌入模型的推荐一致 | `metric_type` |
| 索引类型 | 精确检索（FLAT）还是近似检索（HNSW、IVF 等） | `index_params` 里的 `index_type` |
| 索引参数 | HNSW：建索引时的 `M`、`efConstruction`，检索时的 `ef`；IVF：建索引时的 `nlist`，检索时的 `nprobe`。检索参数越大，结果越准，也越慢 | `index_params`、`search_params` |
| top-k | 返回几条 | `limit` |
| 过滤条件 | 按元数据先筛一遍 | `filter` |
| 返回字段 | 结果里带哪些字段 | `output_fields` |

**几个数据库的原生方法对照**：

| 操作 | Milvus（pymilvus） | Chroma | Qdrant | PGVector（SQL） |
|---|---|---|---|---|
| 建集合 | `create_collection(name, dimension, metric_type)` | `create_collection(name, configuration={"hnsw": {"space": "cosine"}})` | `create_collection(name, vectors_config=VectorParams(size, distance))` | `CREATE TABLE ... (embedding vector(2048))` |
| 相似度度量 | `COSINE` / `L2` / `IP` | `cosine` / `l2` / `ip` | `Distance.COSINE` / `EUCLID` / `DOT` | 运算符 `<=>` 余弦距离、`<->` L2 距离、`<#>` 负内积 |
| 索引 | `AUTOINDEX`、`HNSW`、`IVF_FLAT` 等 | 自动建 HNSW | 自动建 HNSW | `CREATE INDEX ... USING hnsw` 或 `ivfflat` |
| 写入 | `insert` / `upsert(data=[字典])` | `add` / `upsert(ids, embeddings, documents, metadatas)` | `upsert(points=[PointStruct(id, vector, payload)])` | `INSERT` |
| 按 id 取 | `get(ids)` | `get(ids)` | `retrieve(ids)` | `SELECT ... WHERE id = ...` |
| 相似度检索 | `search(data, limit, filter)` | `query(query_embeddings, n_results, where)` | `query_points(query, limit, query_filter)` | `ORDER BY embedding <=> 查询向量 LIMIT k` |
| 过滤写法 | 字符串 `'source == "demo"'` | 字典 `where={"source": "demo"}` | `Filter(must=[FieldCondition(key="source", match=MatchValue(value="demo"))])` | `WHERE` 子句 |
| 删除 | `delete(ids 或 filter)` | `delete(ids 或 where)` | `delete(points_selector)` | `DELETE` |
| 返回的分数 | COSINE、IP 越大越像；L2 越小越像 | 距离，越小越像（余弦距离 = 1 − 余弦相似度） | 相似度，越大越像 | 距离，越小越像 |

可以看出：概念相同，方法名、参数名、过滤写法、分数的方向各不一样。Chroma 一列是用本项目装的 chromadb 1.5.9 实测的，Qdrant 和 PGVector 两列来自它们的官方文档。

#### 2.5.4.2 LangChain 的统一接口：`VectorStore`

在 LangChain 里一般不直接调这些原生 API，而是用各数据库的集成类（2.5.2 表格的最后一列）。它们都继承 `langchain_core.vectorstores.VectorStore`，方法名和参数是统一的。官方文档列出的核心是三个：`add_documents`、`delete`、`similarity_search`。

`VectorStore` 的方法（`langchain_core/vectorstores/base.py`）：

| 方法 | 主要参数 | 返回值 | 说明 |
|---|---|---|---|
| `add_documents(documents, ids=None)` | `list[Document]`，可以给 id | `list[str]`，写入的 id | 内部调用 `embed_documents` 算向量（2.4.4），再连同 `page_content`、`metadata` 一起写入 |
| `add_texts(texts, metadatas=None, ids=None)` | 字符串列表 | `list[str]` | 同上，传的是字符串 |
| `from_documents(documents, embedding, **kwargs)`、`from_texts(...)` | 类方法，`kwargs` 是这个库的构造参数 | 新的向量库对象 | 创建 + 写入一步完成 |
| `delete(ids=None)` | id 列表 | `True`/`False`；没实现时返回 `None` | |
| `get_by_ids(ids)` | id 列表 | `list[Document]` | 父类默认抛 `NotImplementedError`，不是每个库都实现了 |
| `similarity_search(query, k=4, **kwargs)` | 查询文本、条数；过滤等参数放在 `kwargs` 里 | `list[Document]` | 内部先 `embed_query`，再检索 |
| `similarity_search_with_score(query, k=4, **kwargs)` | 同上 | `list[tuple[Document, float]]` | 分数是各库原始的分数，含义各不相同 |
| `similarity_search_with_relevance_scores(query, k=4, score_threshold=None)` | 同上，加一个阈值 | `list[tuple[Document, float]]` | 用 `_select_relevance_score_fn` 把分数换算到 0～1，越大越相关；低于 `score_threshold` 的丢掉 |
| `similarity_search_by_vector(embedding, k=4)` | 查询向量 | `list[Document]` | 已经有向量时用，不再调嵌入模型 |
| `max_marginal_relevance_search(query, k=4, fetch_k=20, lambda_mult=0.5)` | | `list[Document]` | MMR：先取 `fetch_k` 条最相似的，再从中挑出 `k` 条彼此不太重复的；`lambda_mult` 越小越看重多样性 |
| `as_retriever(**kwargs)` | | `VectorStoreRetriever` | 包装成检索器（Retriever），后面学检索器时再展开 |

每个方法都有 `a` 开头的异步版本，例如 `aadd_documents`、`asimilarity_search`。

用同一段代码操作两个库：LangChain 自带的 `InMemoryVectorStore`，和 `langchain-milvus` 的 `Milvus`（另开一个 Milvus Lite 文件，不影响上面的 `rag_tutorial.db`）：

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_milvus import Milvus

documents = [Document(page_content=t, metadata={"source": "demo", "page": i}) for i, t in enumerate(texts)]
ids = ["doc0", "doc1", "doc2", "doc3"]

stores = {
    "InMemoryVectorStore": InMemoryVectorStore(embedding=embedding_model),
    "Milvus": Milvus(
        embedding_function=embedding_model,
        connection_args={"uri": "db/langchain_demo.db"},
        collection_name="lc_docs",
        index_params={"index_type": "AUTOINDEX", "metric_type": "COSINE"},
        consistency_level="Strong",  # Milvus Lite 只支持 Strong，langchain-milvus 默认是 Session
        drop_old=True,               # 已有同名 collection 时先删掉，重跑结果一样
    ),
}

for name, vs in stores.items():
    print(f"{name}.add_documents →", vs.add_documents(documents, ids=ids))

InMemoryVectorStore.add_documents → ['doc0', 'doc1', 'doc2', 'doc3']


Milvus.add_documents → ['doc0', 'doc1', 'doc2', 'doc3']


In [ ]:
query = "什么是langchain？"
for name, vs in stores.items():
    print(f"== {name}")
    for doc, score in vs.similarity_search_with_score(query, k=2):
        print(f"  {score:.4f}  {doc.page_content}")

== InMemoryVectorStore


  0.8585  LangChain 是一个用于构建 LLM 应用的开发框架。
  0.5008  RAG 的核心是先检索相关知识，再让大模型生成答案。
== Milvus


  0.8585  LangChain 是一个用于构建 LLM 应用的开发框架。
  0.5008  RAG 的核心是先检索相关知识，再让大模型生成答案。


写入和检索的代码完全一样，换数据库只改了创建的那一行。两个库都用余弦相似度，分数也和 2.5.3.5 的 `search` 一样（0.8585、0.5008）。

但“统一接口”只统一了方法名，下面这些地方各库仍然不同：

In [ ]:
inmem, milvus_store = stores["InMemoryVectorStore"], stores["Milvus"]

# 1. 过滤：参数名和写法不同
docs_a = inmem.similarity_search(query, k=2, filter=lambda doc: doc.metadata["page"] >= 2)
docs_b = milvus_store.similarity_search(query, k=2, expr="page >= 2")
for name, docs in [("InMemoryVectorStore", docs_a), ("Milvus", docs_b)]:
    print(name)
    for doc in docs:
        print(f"  id={doc.id!r}  metadata={doc.metadata}  {doc.page_content}")

InMemoryVectorStore
  id='doc2'  metadata={'source': 'demo', 'page': 2}  RAG 的核心是先检索相关知识，再让大模型生成答案。
  id='doc3'  metadata={'source': 'demo', 'page': 3}  Docker Desktop 可以方便地在本地运行 Milvus Standalone。
Milvus
  id=None  metadata={'pk': 'doc2', 'source': 'demo', 'page': 2}  RAG 的核心是先检索相关知识，再让大模型生成答案。
  id=None  metadata={'pk': 'doc3', 'source': 'demo', 'page': 3}  Docker Desktop 可以方便地在本地运行 Milvus Standalone。


In [ ]:
def try_call(label, fn):
    try:
        print(f"  {label} → {fn()}")
    except Exception as e:
        print(f"  {label} → {type(e).__name__}: {e}")

# 2. 有的方法没有实现；3. 返回值不同
for name, vs in stores.items():
    print(name)
    try_call("get_by_ids", lambda: [d.page_content for d in vs.get_by_ids(["doc0"])])
    try_call("similarity_search_with_relevance_scores",
             lambda: [round(s, 4) for _, s in vs.similarity_search_with_relevance_scores(query, k=2)])
    try_call("delete", lambda: vs.delete(ids=["doc3"]))

# 4. Milvus 的过滤参数不叫 filter
print("Milvus")
try_call("similarity_search(filter=...)", lambda: milvus_store.similarity_search(query, k=2, filter="page >= 2"))

InMemoryVectorStore
  get_by_ids → ['LangChain 是一个用于构建 LLM 应用的开发框架。']
  similarity_search_with_relevance_scores → NotImplementedError: 
  delete → None
Milvus
  get_by_ids → NotImplementedError: Milvus does not yet support get_by_ids.


  similarity_search_with_relevance_scores → [0.9293, 0.7504]
  delete → True
Milvus


  similarity_search(filter=...) → TypeError: pymilvus.milvus_client.milvus_client.MilvusClient.search() got multiple values for keyword argument 'filter'


| 区别 | `InMemoryVectorStore` | `Milvus`（langchain-milvus 0.4.0） |
|---|---|---|
| 构造参数里的嵌入模型 | `embedding` | `embedding_function` |
| 过滤 | `filter=函数`，函数接收一个 `Document`，返回 `True` 时保留 | `expr="page >= 2"`，就是 2.5.3.5 的过滤表达式。写成 `filter=` 会报 `TypeError`：源码自己已经把 `expr` 作为 `filter` 传给 `MilvusClient.search`，再传一个就重复了 |
| 返回的 `Document` 的 id | 在 `doc.id` 里 | `doc.id` 是 `None`，id 在 `doc.metadata["pk"]` 里（主键字段默认叫 `pk`） |
| `get_by_ids` | 支持 | 没实现，抛 `NotImplementedError` |
| `similarity_search_with_score` 的分数 | 余弦相似度 | 原始的度量值：COSINE 是余弦相似度，L2 是距离（越小越像） |
| `similarity_search_with_relevance_scores` | 没实现 `_select_relevance_score_fn`，抛 `NotImplementedError` | COSINE、IP 用 `(分数 + 1) / 2` 换算到 0～1，L2 用 `1 − 分数 / 4` |
| `delete` 的返回值 | `None` | `True` |
| 主键重复时 | `add_documents` 覆盖 | `add_documents` 底层调的是 `MilvusClient.insert`，服务端会出现重复数据；更新用 `Milvus.upsert(ids, documents)` |
| 相似度度量 | 固定是余弦相似度 | `index_params` 的 `metric_type`，不传时默认 `L2` |

其他常见集成的写法（来自官方文档）：

| | Chroma | PGVector | Qdrant |
|---|---|---|---|
| 包 / 类 | `langchain-chroma` / `Chroma` | `langchain-postgres` / `PGVector` | `langchain-qdrant` / `QdrantVectorStore` |
| 嵌入模型参数 | `embedding_function` | `embeddings` | `embedding` |
| 数据存在哪 | `persist_directory="./chroma_db"`（不传就在内存里） | `connection="postgresql+psycopg://..."` | `client=QdrantClient(...)` |
| 过滤 | `filter={"source": "demo"}` | `filter={"source": {"$eq": "demo"}}`，支持 `$in`、`$gt`、`$like`、`$and` 等 | `filter=models.Filter(...)` |

所以换数据库时，`add_documents`、`similarity_search`、`delete` 的基本用法不用改；构造参数、过滤条件、分数的含义要按新库的文档改。

#### 补充：两种库的数据存在哪里

**InMemoryVectorStore：只在内存里**

源码（`langchain_core/vectorstores/in_memory.py`）里，整个库就是一个字典：

```python
self.store: dict[str, dict[str, Any]] = {}

# add_documents 时，每条写成：
self.store[doc_id] = {"id": doc_id, "vector": vector, "text": doc.page_content, "metadata": doc.metadata}
```

* 检索时，把查询向量和字典里的每一条都算一遍余弦相似度。没有索引，相当于 FLAT 暴力检索，数据多了会很慢；
* 数据在 Python 进程的内存里，重启 kernel 就没了；
* 它提供了 `dump(path)`，把这个字典存成 JSON 文件，再用 `InMemoryVectorStore.load(path, embedding)` 读回来。这只是手动导出和导入：

In [ ]:
inmem.dump("db/inmemory_demo.json")

reloaded = InMemoryVectorStore.load("db/inmemory_demo.json", embedding=embedding_model)
print("读回的 id：", list(reloaded.store))
record = reloaded.store["doc0"]
print("每条的键：", list(record), "，向量长度：", len(record["vector"]))

读回的 id： ['doc0', 'doc1', 'doc2']
每条的键： ['id', 'vector', 'text', 'metadata'] ，向量长度： 2048


上面删掉了 `doc3`，所以读回 3 条。每条存的就是源码里那 4 个键。

**Milvus Lite：存在磁盘上的一个文件夹里**

`MilvusClient("db/rag_tutorial.db")` 里的 `.db` 只是让 pymilvus 认出“这是 Milvus Lite”的后缀（2.5.3.3）。milvus-lite 3.x 把这个路径当作数据**文件夹**，看一下里面有什么：

In [ ]:
from pathlib import Path

root = Path("db/rag_tutorial.db")
for p in sorted(root.rglob("*")):
    if p.is_file():
        print(p.relative_to(root))
    elif not any(p.iterdir()):
        print(f"{p.relative_to(root)}/  （空文件夹）")

.database-staging/  （空文件夹）
LOCK
collections/  （空文件夹）
databases/rag_tutorial/collections/docs/manifest.json
databases/rag_tutorial/collections/docs/manifest.json.prev
databases/rag_tutorial/collections/docs/partitions/_default/data/data_000001_000004.parquet
databases/rag_tutorial/collections/docs/partitions/_default/indexes/data_000001_000004.vector.autoindex.idx
databases/rag_tutorial/collections/docs/schema.json
databases/rag_tutorial/collections/docs/wal/  （空文件夹）
databases/rag_tutorial/database_properties.json


各部分的作用（milvus-lite 3.2.1 源码，`milvus_lite/db.py`、`milvus_lite/storage/`）：

| 路径 | 作用 |
|---|---|
| `LOCK` | 文件锁。打开这个文件夹的进程会一直持有它，所以同一时间只能有一个进程打开，另一个 kernel 再打开会报 `DataDirLockedError` |
| `collections/`（最外层） | `default` 库的 collection，这里是空的 |
| `.database-staging/` | 创建数据库时的临时文件夹，没建完的下次启动时清理 |
| `databases/rag_tutorial/` | 2.5.3.3 创建的 `rag_tutorial` 库 |
| `.../docs/schema.json` | `docs` 的字段定义 |
| `.../docs/manifest.json` | 记录当前有哪些数据文件；`manifest.json.prev` 是上一个版本，`manifest.json` 损坏时用它恢复 |
| `.../data/data_000001_000004.parquet` | 那 4 条数据（`id`、`vector`、`$meta`），Parquet 列式文件。文件名里的两个数字是这批数据的序号范围 |
| `.../indexes/...autoindex.idx` | 向量字段的索引文件，装了 faiss-cpu 时是 HNSW |
| `.../docs/wal/` | 预写日志（WAL），还没 flush 的写入记在这里，现在是空的 |
| `.../delta/`（运行过 `delete` 才有） | 记录哪些主键被删了。删除不改原来的数据文件 |

数据的写入和检索过程：

1. **写入**：先追加到 `wal/` 里的日志文件，再放进内存表，这时 `search`、`query` 就能查到；
2. **flush**：内存表满 10000 条、手动 `flush`，或者进程正常退出时（关闭时会做最后一次 flush），内存表写成 `data/` 里的 Parquet 文件并建好索引，对应的日志删除；
3. **进程意外结束**（比如强制停止 kernel）：没 flush 的数据还在 `wal/` 里，下次打开时重放日志恢复；
4. **检索**：`load_collection` 把数据文件和索引读进内存，检索在内存里做。所以 Lite 运行时也占内存，只是数据以磁盘上的文件为准。重启 kernel 后数据还在，重新 load 就能查（2.5.3.3 里 released 的报错就是少了这一步）。

另外，Lite 服务是在当前 Python 进程里以后台线程运行的（`milvus_lite/server_manager.py`），pymilvus 再通过本机的 gRPC 端口连上它。旧版 milvus-lite 是单独启动一个 C++ 进程。

| | InMemoryVectorStore | Milvus Lite |
|---|---|---|
| 数据在哪 | Python 进程内存里的一个字典 | 磁盘上的文件夹（数据文件 + 索引 + WAL），检索时载入内存 |
| 重启 kernel 后 | 没了，除非自己 `dump` | 还在，重新 load 即可 |
| 索引 | 没有，逐条计算 | AUTOINDEX（HNSW） |
| 多个进程同时用 | 每个进程各有一份，互不相干 | 同一时间只能一个进程打开（`LOCK`） |